In [0]:
from pyspark.sql.functions import col, current_timestamp
from delta.tables import DeltaTable

# 1. Pipeline Storage Configurations
source_directory = "/Volumes/mde_dev/bronze/raw_landing/"  # or ADLS/S3 path
checkpoint_path = "/Volumes/mde_dev/bronze/_checkpoints/auto_loader_upsert/"
target_table_name = "mde_dev.bronze.orders_ingested"

# 2. Configure Auto Loader Stream to Track New and Modified Files
# Using cloudFiles.allowOverwrites forces Auto Loader to re-read files if modified
raw_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")               # parquet, json, csv, etc.
    .option("cloudFiles.allowOverwrites", "true")         # Re-read files when modified/overwritten
    .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
    .load(source_directory)
    .select(
        "*",
        # Capture file metadata to track origin and updates
        col("_metadata.file_path").alias("ingested_file_path"),
        col("_metadata.file_modification_time").alias("file_modification_time"),
        current_timestamp().alias("processed_at")
    )
)

# 3. Define the Micro-Batch Upsert Handler
def upsert_files_to_delta(micro_batch_df, batch_id):
    """
    If a file is new, its rows are appended.
    If an existing file was modified, its prior rows in the target table are
    overwritten with the newest version of that file.
    """
    if micro_batch_df.isEmpty():
        return

    # Create target table automatically on first batch if it does not exist
    if not spark.catalog.tableExists(target_table_name):
        (
            micro_batch_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(target_table_name)
        )
        return

    target_delta = DeltaTable.forName(spark, target_table_name)

    # Strategy A: Overwrite at the FILE LEVEL using file_path
    # (Removes old rows from that file and replaces them with the new file contents)
    (
        target_delta.alias("target")
        .merge(
            micro_batch_df.alias("source"),
            # Matches records originating from the exact same file path
            "target.ingested_file_path = source.ingested_file_path AND target.id = source.id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

# 4. Start the Streaming Query
query = (
    raw_stream_df.writeStream
    .format("delta")
    .foreachBatch(upsert_files_to_delta)
    .option("checkpointLocation", f"{checkpoint_path}/stream")
    .trigger(availableNow=True)  # Set to processingTime='1 minute' for continuous stream
    .start()
)

query.awaitTermination()

## Alternative: Whole-File Replacement (When Files Do Not Have Record IDs)

If the modified files do not have row-level primary keys (e.g., standard flat dumps), replace the merge block inside upsert_files_to_delta with a file-level deletion before appending the latest data

In [0]:
def replace_modified_files(micro_batch_df, batch_id):
    if micro_batch_df.isEmpty():
        return

    if not spark.catalog.tableExists(target_table_name):
        micro_batch_df.write.format("delta").mode("append").saveAsTable(target_table_name)
        return

    # Identify all distinct files present in this incoming micro-batch
    distinct_files = [
        row["ingested_file_path"] 
        for row in micro_batch_df.select("ingested_file_path").distinct().collect()
    ]
    
    # Format list for SQL deletion
    files_sql = ", ".join([f"'{f}'" for f in distinct_files])
    
    # Atomically delete previous rows that originated from these specific files
    spark.sql(f"""
        DELETE FROM {target_table_name}
        WHERE ingested_file_path IN ({files_sql})
    """)
    
    # Append the incoming new/modified file content
    micro_batch_df.write.format("delta").mode("append").saveAsTable(target_table_name)